In [4]:
import datetime
from pathlib import Path
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# --- Setup paths ---
ASIPdir = Path("D:/MyDrive/Stability/RawData/Monthly_Averages/MergedOutput_month_year")
ingtdir = ASIPdir / "_GeoTIFFs_100m_gdal"
outdir = ASIPdir / "BOEM_GeoTIFFs"
beaudir = outdir / "Beau"
chukdir = outdir / "Chuk"

for d in [outdir, beaudir, chukdir]:
    d.mkdir(parents=True, exist_ok=True)

nowtime = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
croplog = outdir / f"croplog_{nowtime}.txt"

TARGET_CRS = "EPSG:3338"

# --- Master GeoTIFFs ---
beau_master = Path("D:/MyDrive/ASIP_IceCharts/BOEM_GeoTIFFs/Beau/2017-18/a2017_246-246_slie.tif")
chuk_master = Path("D:/MyDrive/ASIP_IceCharts/BOEM_GeoTIFFs/Chuk/2017-18/a2017_246-246_slie.tif")

with rasterio.open(beau_master) as src:
    beau_transform = src.transform
    beau_width = src.width
    beau_height = src.height
    beau_crs = src.crs

with rasterio.open(chuk_master) as src:
    chuk_transform = src.transform
    chuk_width = src.width
    chuk_height = src.height
    chuk_crs = src.crs

# --- Mask creation ---
def create_mask(src_file, dst_file):
    with rasterio.open(src_file) as src:
        arr = src.read(1)
        mask = np.where((arr == 255) | (arr == 128), arr, 0)
        profile = src.profile
        profile.update(dtype=rasterio.uint8, nodata=0, compress="packbits")
        with rasterio.open(dst_file, "w", **profile) as dst:
            dst.write(mask.astype(np.uint8), 1)
    # Ensure CRS is set
    with rasterio.open(dst_file, "r+") as ds:
        if ds.crs is None:
            ds.crs = TARGET_CRS

# --- Apply mask ---
def apply_mask(data, profile, mask_file):
    with rasterio.open(mask_file) as msk:
        mask_data = msk.read(1)
        reprojected_mask = np.zeros((profile["height"], profile["width"]), dtype=np.uint8)
        reproject(
            source=mask_data,
            destination=reprojected_mask,
            src_transform=msk.transform,
            src_crs=msk.crs,
            dst_transform=profile["transform"],
            dst_crs=profile["crs"],
            resampling=Resampling.nearest
        )

    # Convert raster to float32 for scaling
    data_scaled = (data.astype(np.float32) * 255).astype(np.uint8)

    # Apply mask: land=128, sea=0, ice/other=data*255
    masked = np.where(reprojected_mask == 255, 0,
              np.where(reprojected_mask == 128, 128, data_scaled))
    return masked

# --- Reproject + align to master grid ---
def reproject_crop_mask_master(src_file, dst_file, master_transform, master_width, master_height, master_crs, mask_file):
    with rasterio.open(src_file) as src:
        profile = src.profile.copy()
        profile.update({
            "crs": master_crs,
            "transform": master_transform,
            "width": master_width,
            "height": master_height,
            "dtype": np.uint8,
            "nodata": None,
            "compress": "packbits"
        })

        data = np.zeros((src.count, master_height, master_width), dtype=np.uint8)

        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=data[i-1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=master_transform,
                dst_crs=master_crs,
                resampling=Resampling.nearest,
                src_nodata=src.nodata,
                dst_nodata=None
            )

        masked = apply_mask(data, profile, mask_file)

        with rasterio.open(dst_file, "w", **profile) as dst:
            dst.write(masked)

# --- Helper: build output filename ---
def build_outname(tif_name: Path) -> str:
    # Extract YYYY and MM from input
    parts = tif_name.stem.split("_")  # ["WeightedMean", "YYYY", "MM"]
    year = int(parts[1])
    month = int(parts[2])

    # First day of month
    first_day = datetime.date(year, month, 1)
    # Second day
    second_day = first_day + datetime.timedelta(days=1)

    # Format MMDD
    first_str = first_day.strftime("%m%d")
    second_str = second_day.strftime("%m%d")

    return f"i{first_str}_{second_str}_{year}_Coh.tif"


# --- MAIN IMPLEMENTATION ---
if __name__ == "__main__":
    # Ensure masks exist
    beau_mask = beaudir / "beau_landmask.tif"
    if not beau_mask.exists():
        src_mask = Path("D:/MyDrive/EM2025_2025/ConfigData/Beaufort/Landmask_Beaufort_4col.tif")
        create_mask(src_mask, beau_mask)

    chuk_mask = chukdir / "chuk_landmask.tif"
    if not chuk_mask.exists():
        src_mask = Path("D:/MyDrive/EM2025_2025/ConfigData/Chukchi/Landmask_Chukchi_4col.tif")
        create_mask(src_mask, chuk_mask)

    # Process all input GeoTIFFs
    with open(croplog, "w") as log:
        for tif in ingtdir.glob("*.tif"):
            outname = build_outname(tif)

            # Beaufort
            beau_out = beaudir / outname
            reproject_crop_mask_master(tif, beau_out, beau_transform, beau_width, beau_height, beau_crs, beau_mask)
            log.write(f"{outname}\n")

            # Chukchi
            chuk_out = chukdir / outname
            reproject_crop_mask_master(tif, chuk_out, chuk_transform, chuk_width, chuk_height, chuk_crs, chuk_mask)
            log.write(f"{outname}\n")

            print(f"{tif.name} -> {outname} done")

WeightedMean_2017_03.tif -> 0301_0302_2017_Coh.tif done
WeightedMean_2017_04.tif -> 0401_0402_2017_Coh.tif done
WeightedMean_2017_05.tif -> 0501_0502_2017_Coh.tif done
WeightedMean_2017_06.tif -> 0601_0602_2017_Coh.tif done
WeightedMean_2017_07.tif -> 0701_0702_2017_Coh.tif done
WeightedMean_2017_08.tif -> 0801_0802_2017_Coh.tif done
WeightedMean_2017_09.tif -> 0901_0902_2017_Coh.tif done
WeightedMean_2017_10.tif -> 1001_1002_2017_Coh.tif done
WeightedMean_2017_11.tif -> 1101_1102_2017_Coh.tif done
WeightedMean_2017_12.tif -> 1201_1202_2017_Coh.tif done
WeightedMean_2018_01.tif -> 0101_0102_2018_Coh.tif done
WeightedMean_2018_02.tif -> 0201_0202_2018_Coh.tif done
WeightedMean_2018_03.tif -> 0301_0302_2018_Coh.tif done
WeightedMean_2018_04.tif -> 0401_0402_2018_Coh.tif done
WeightedMean_2018_05.tif -> 0501_0502_2018_Coh.tif done
WeightedMean_2018_06.tif -> 0601_0602_2018_Coh.tif done
WeightedMean_2018_07.tif -> 0701_0702_2018_Coh.tif done
WeightedMean_2018_08.tif -> 0801_0802_2018_Coh.t